<a href="https://colab.research.google.com/github/mufadah/2026_VisiKomputer/blob/main/Jobsheet_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 01 - Inisialisasi Kamera dan Akuisisi Citra
**Tujuan:** Mahasiswa mampu menginisialisasi kamera dan menampilkan citra secara *real-time* sebagai dasar untuk pengolahan visual selanjutnya.

**Indikator Keberhasilan:** Citra tampil dengan stabil, FPS berada di atas 10, dan proses dapat dihentikan dengan menekan tombol `q` atau menutup jendela preview tanpa error.

In [ ]:
# Instalasi pustaka pendukung jika belum terpasang:
# !pip install opencv-python cvzone mediapipe numpy

import cv2
import time

# Inisialisasi kamera bawaan laptop (indeks 0 dengan backend DirectShow / cv2.CAP_DSHOW)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka. Pastikan izin kamera aktif dan tidak sedang digunakan aplikasi lain (Zoom, Meet, Camera, dll).")

window_name = "Preview"
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
frames, t0 = 0, time.time()

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames += 1
        
        # Hitung dan perbarui FPS setiap 1 detik pada judul jendela
        if time.time() - t0 >= 1.0:
            cv2.setWindowTitle(window_name, f"Preview (FPS ~ {frames})")
            frames, t0 = 0, time.time()
            
        cv2.imshow(window_name, frame)
        
        # Tekan tombol 'q' atau klik tombol silang (X) pada jendela untuk keluar
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
finally:
    cap.release()
    cv2.destroyAllWindows()


# Praktikum 02 - Deteksi Pose dan Analisis Sudut Tubuh
**Tujuan:** Mahasiswa mampu menerapkan *pose estimation* menggunakan MediaPipe/cvzone dan menghitung sudut sendi berdasarkan koordinat landmark tubuh menggunakan geometri vektor.

**Landmark Kunci:**
- 11: Bahu kiri (*left shoulder*)
- 13: Siku kiri (*left elbow*)
- 15: Pergelangan tangan kiri (*left wrist*)

**Indikator Keberhasilan:** Kerangka tubuh terdeteksi dengan baik dan nilai sudut sendi siku berubah secara akurat sesuai pergerakan.

In [ ]:
import cv2
import numpy as np
from cvzone.PoseModule import PoseDetector

# Inisialisasi video capture webcam
# Inisialisasi kamera bawaan laptop (indeks 0 dengan backend DirectShow / cv2.CAP_DSHOW)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka. Pastikan izin kamera aktif dan tidak sedang digunakan aplikasi lain (Zoom, Meet, Camera, dll).")

# Inisialisasi PoseDetector dari cvzone
detector = PoseDetector(
    staticMode=False,
    modelComplexity=1,
    enableSegmentation=False,
    detectionCon=0.5,
    trackCon=0.5
)

window_name = "Pose + Angle"
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)

try:
    while True:
        success, img = cap.read()
        if not success:
            break

        # Temukan pose manusia dalam frame
        img = detector.findPose(img)

        # Temukan landmark, bounding box, dan pusat tubuh dalam frame
        lmList, bboxInfo = detector.findPosition(img, draw=True, bboxWithHands=False)

        # Periksa apakah ada landmark tubuh yang terdeteksi
        if lmList:
            # Dapatkan pusat bounding box di sekitar tubuh
            center = bboxInfo["center"]

            # Gambar lingkaran di pusat bounding box
            cv2.circle(img, center, 5, (255, 0, 255), cv2.FILLED)

            # Hitung jarak antara landmark 11 (bahu) dan 15 (pergelangan tangan)
            length, img, info = detector.findDistance(
                lmList[11][0:2],
                lmList[15][0:2],
                img=img,
                color=(255, 0, 0),
                scale=10
            )

            # Hitung sudut siku antara landmark 11, 13, dan 15
            angle, img = detector.findAngle(
                lmList[11][0:2],
                lmList[13][0:2],
                lmList[15][0:2],
                img=img,
                color=(0, 0, 255),
                scale=10
            )

            # Periksa apakah sudut mendekati 50 derajat dengan offset 10 (rentang 40 - 60 derajat)
            isCloseAngle50 = detector.angleCheck(myAngle=angle, targetAngle=50, offset=10)
            print(f"Sudut Siku: {angle:.2f} | Mendekati 50 deg: {isCloseAngle50}")

        cv2.imshow(window_name, img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
finally:
    cap.release()
    cv2.destroyAllWindows()


# Praktikum 03 - Deteksi Wajah dan Analisis Kedipan Mata
**Tujuan:** Mahasiswa mampu mendeteksi wajah menggunakan MediaPipe Face Mesh dan mengukur perubahan jarak antar landmark mata untuk mendeteksi kedipan mata secara otomatis berdasarkan konsep *Eye Aspect Ratio* (EAR).

**Formulasi EAR:**
$$\text{EAR} = \frac{d_{\text{vertikal}}}{d_{\text{horizontal}}}$$

Landmark mata kiri yang digunakan: Vertikal (159, 145), Horizontal (33, 133).
Saat mata terpejam, jarak vertikal mengecil drastis sehingga nilai EAR turun di bawah ambang batas (*threshold*).

In [ ]:
import cv2
import numpy as np
from cvzone.FaceMeshModule import FaceMeshDetector

# Indeks landmark mata kiri: vertikal (159, 145), horizontal (33, 133)
L_TOP, L_BOTTOM, L_LEFT, L_RIGHT = 159, 145, 33, 133

def dist(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

# Inisialisasi kamera bawaan laptop (indeks 0 dengan backend DirectShow / cv2.CAP_DSHOW)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka. Pastikan izin kamera aktif dan tidak sedang digunakan aplikasi lain (Zoom, Meet, Camera, dll).")

# Inisialisasi FaceMeshDetector (mendeteksi hingga 2 wajah)
detector = FaceMeshDetector(
    staticMode=False,
    maxFaces=2,
    minDetectionCon=0.5,
    minTrackCon=0.5
)

# Parameter dan variabel penghitung kedipan mata
blink_count = 0
closed_frames = 0
CLOSED_FRAMES_THRESHOLD = 3   # Jumlah frame berurutan untuk dianggap kedipan valid
EYE_AR_THRESHOLD = 0.20        # Ambang batas EAR untuk mendeteksi mata tertutup
is_closed = False

window_name = "FaceMesh + EAR"
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)

try:
    while True:
        ok, img = cap.read()
        if not ok:
            break
            
        img, faces = detector.findFaceMesh(img, draw=True)
        if faces:
            face = faces[0]  # List koordinat 468 titik landmark wajah (x, y)
            v = dist(face[L_TOP][:2], face[L_BOTTOM][:2])
            h = dist(face[L_LEFT][:2], face[L_RIGHT][:2])
            ear = v / (h + 1e-8)
            
            # Tampilkan nilai EAR di layar
            cv2.putText(
                img, f"EAR(L): {ear:.3f}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2
            )

            # Logika deteksi kedipan: jika EAR < threshold selama >= 3 frame berturut-turut
            if ear < EYE_AR_THRESHOLD:
                closed_frames += 1
                if closed_frames >= CLOSED_FRAMES_THRESHOLD and not is_closed:
                    blink_count += 1
                    is_closed = True
            else:
                closed_frames = 0
                is_closed = False

            # Tampilkan total kedipan pada frame
            cv2.putText(
                img, f"Blink: {blink_count}", (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2
            )

        cv2.imshow(window_name, img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
finally:
    cap.release()
    cv2.destroyAllWindows()


# Praktikum 04 - Deteksi Tangan dan Penghitungan Jumlah Jari
**Tujuan:** Mahasiswa mampu mendeteksi tangan manusia dan menghitung jumlah jari yang terangkat menggunakan modul Hand Tracking berbasis MediaPipe.

**Logika Deteksi Jari:**
Membandingkan posisi koordinat vertikal (y) antara ujung jari (*tip*) dan sendi bawahnya (*PIP*). Jika $y_{\text{tip}} < y_{\text{pip}}$, maka jari dinyatakan dalam kondisi terbuka/terangkat.

In [ ]:
import cv2
from cvzone.HandTrackingModule import HandDetector

# Inisialisasi kamera bawaan laptop (indeks 0 dengan backend DirectShow / cv2.CAP_DSHOW)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka. Pastikan izin kamera aktif dan tidak sedang digunakan aplikasi lain (Zoom, Meet, Camera, dll).")

# Inisialisasi HandDetector
detector = HandDetector(
    staticMode=False,
    maxHands=1,
    modelComplexity=1,
    detectionCon=0.5,
    minTrackCon=0.5
)

window_name = "Hands + Fingers"
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)

try:
    while True:
        ok, img = cap.read()
        if not ok:
            break

        # Deteksi tangan (flipType=True untuk orientasi cermin / mirror UI)
        hands, img = detector.findHands(img, draw=True, flipType=True)
        if hands:
            hand = hands[0]  # Dictionary landmark tangan
            fingers = detector.fingersUp(hand)  # List 5 elemen [ibu_jari, telunjuk, tengah, manis, kelingking]
            count = sum(fingers)                # Hitung jumlah jari yang bernilai 1
            
            cv2.putText(
                img, f"Fingers: {count} {fingers}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2
            )

        cv2.imshow(window_name, img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
finally:
    cap.release()
    cv2.destroyAllWindows()


# Praktikum 05 - Pengenalan Gestur Tangan (Hand Gesture Recognition)
**Tujuan:** Mahasiswa mampu mengenali gestur tangan sederhana (*Thumbs Up*, *OK*, dan *Rock-Paper-Scissors*) berdasarkan hubungan geometris antar landmark tangan.

**Kriteria Geometris Gestur:**
- **OK:** Jarak antara ujung ibu jari (4) dan ujung telunjuk (8) < 35 piksel.
- **THUMBS_UP:** Ibu jari mengarah ke atas ($y$ kecil) dan menjauh dari pergelangan tangan.
- **ROCK:** Rata-rata jarak seluruh ujung jari ke pergelangan tangan (*wrist*) < 120 piksel (mengepal).
- **PAPER:** Rata-rata jarak seluruh ujung jari ke pergelangan tangan > 200 piksel (membuka lebar).
- **SCISSORS:** Jari telunjuk dan tengah terbuka (>180 px), sedangkan jari manis dan kelingking tertutup (<160 px).

In [ ]:
import cv2
import numpy as np
from cvzone.HandTrackingModule import HandDetector

def dist(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def classify_gesture(hand):
    # hand["lmList"] berisi 21 landmark titik tangan (x, y, z)
    lm = hand["lmList"]
    wrist = np.array(lm[0][:2])
    thumb_tip = np.array(lm[4][:2])
    index_tip = np.array(lm[8][:2])
    middle_tip = np.array(lm[12][:2])
    ring_tip = np.array(lm[16][:2])
    pinky_tip = np.array(lm[20][:2])

    # Rata-rata jarak relatif ujung jari ke pergelangan tangan (wrist)
    r_mean = np.mean([
        dist(index_tip, wrist),
        dist(middle_tip, wrist),
        dist(ring_tip, wrist),
        dist(pinky_tip, wrist),
        dist(thumb_tip, wrist)
    ])

    # 1. Gestur OK
    if dist(thumb_tip, index_tip) < 35:
        return "OK"

    # 2. Gestur Thumbs Up
    if (thumb_tip[1] < wrist[1] - 40) and (dist(thumb_tip, wrist) > 0.8 * dist(index_tip, wrist)):
        return "THUMBS_UP"

    # 3. Gestur Rock (Batu/Kepalan)
    if r_mean < 120:
        return "ROCK"

    # 4. Gestur Paper (Kertas/Telapak Terbuka)
    if r_mean > 200:
        return "PAPER"

    # 5. Gestur Scissors (Gunting)
    if (dist(index_tip, wrist) > 180 and dist(middle_tip, wrist) > 180 and
        dist(ring_tip, wrist) < 160 and dist(pinky_tip, wrist) < 160):
        return "SCISSORS"

    return "UNKNOWN"

# Inisialisasi kamera bawaan laptop (indeks 0 dengan backend DirectShow / cv2.CAP_DSHOW)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka. Pastikan izin kamera aktif dan tidak sedang digunakan aplikasi lain (Zoom, Meet, Camera, dll).")

detector = HandDetector(
    staticMode=False,
    maxHands=1,
    modelComplexity=1,
    detectionCon=0.5,
    minTrackCon=0.5
)

window_name = "Hand Gestures (cvzone)"
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)

try:
    while True:
        ok, img = cap.read()
        if not ok:
            break

        hands, img = detector.findHands(img, draw=True, flipType=True)
        if hands:
            label = classify_gesture(hands[0])
            cv2.putText(
                img, f"Gesture: {label}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2
            )

        cv2.imshow(window_name, img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
finally:
    cap.release()
    cv2.destroyAllWindows()


# Praktikum 06 - Analisis Gerakan Tubuh dan Penghitung Aktivitas
**Tujuan:** Mahasiswa mampu membuat sistem penghitung aktivitas fisik (*squat* dan *push-up*) menggunakan data landmark pose dan logika kondisi gerakan dengan teknik *debounce*.

**Aturan Gerakan:**
- **Squat:** Menghitung rata-rata sudut lutut kiri (23-25-27) & kanan (24-26-28). Kondisi `down` jika $\theta < 80^\circ$ dan `up` jika $\theta > 160^\circ$.
- **Push-up:** Menggunakan rasio jarak bahu-pergelangan terhadap bahu-pinggul. Kondisi `down` jika rasio $< 0.85$ dan `up` jika rasio $> 1.00$.
- **Debounce:** Memerlukan minimal 4 frame konsisten sebelum state berpindah untuk mencegah repetisi terhitung ganda (*false counting*).
- **Kontrol:** Tekan tombol `m` untuk berpindah mode (*squat* / *pushup*), dan `q` untuk keluar.

In [ ]:
import cv2
import numpy as np
from collections import deque
from cvzone.PoseModule import PoseDetector

MODE = "squat"                 # Tekan 'm' untuk toggle ke "pushup"
KNEE_DOWN, KNEE_UP = 80, 160   # Ambang batas squat (derajat)
DOWN_R, UP_R = 0.85, 1.00      # Ambang batas push-up (rasio)
SAMPLE_OK = 4                  # Minimal frame konsisten sebelum berganti state

# Inisialisasi kamera bawaan laptop (indeks 0 dengan backend DirectShow / cv2.CAP_DSHOW)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("Kamera tidak bisa dibuka. Pastikan izin kamera aktif dan tidak sedang digunakan aplikasi lain (Zoom, Meet, Camera, dll).")

detector = PoseDetector(
    staticMode=False,
    modelComplexity=1,
    enableSegmentation=False,
    detectionCon=0.5,
    trackCon=0.5
)

count, state = 0, "up"
debounce = deque(maxlen=6)

def ratio_pushup(lm):
    # Gunakan sisi kiri: 11=shoulderL, 15=wristL, 23=hipL
    idx = slice(1, 3) if len(lm[11]) >= 4 else slice(0, 2)
    sh = np.array(lm[11][idx])
    wr = np.array(lm[15][idx])
    hp = np.array(lm[23][idx])
    return np.linalg.norm(sh - wr) / (np.linalg.norm(sh - hp) + 1e-8)

window_name = "Pose Counter"
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)

try:
    while True:
        ok, img = cap.read()
        if not ok:
            break

        img = detector.findPose(img, draw=True)
        lmList, _ = detector.findPosition(img, draw=False)
        flag = None

        if lmList:
            if MODE == "squat":
                # Hitung sudut lutut kiri (23, 25, 27) dan lutut kanan (24, 26, 28)
                angL, img = detector.findAngle(
                    lmList[23][0:2],
                    lmList[25][0:2],
                    lmList[27][0:2],
                    img=img,
                    color=(0, 0, 255),
                    scale=10
                )

                angR, img = detector.findAngle(
                    lmList[24][0:2],
                    lmList[26][0:2],
                    lmList[28][0:2],
                    img=img,
                    color=(0, 255, 0),
                    scale=10
                )

                ang = (angL + angR) / 2.0
                if ang < KNEE_DOWN:
                    flag = "down"
                elif ang > KNEE_UP:
                    flag = "up"
                    
                cv2.putText(
                    img, f"Knee: {ang:5.1f}", (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2
                )
            else:
                # Push-up: rasio jarak (shoulder-wrist)/(shoulder-hip)
                r = ratio_pushup(lmList)
                if r < DOWN_R:
                    flag = "down"
                elif r > UP_R:
                    flag = "up"
                    
                cv2.putText(
                    img, f"Ratio: {r:4.2f}", (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2
                )

            # Logika debounce filter untuk mendeteksi repetisi (transisi down -> up)
            debounce.append(flag)
            if debounce.count("down") >= SAMPLE_OK and state == "up":
                state = "down"
            if debounce.count("up") >= SAMPLE_OK and state == "down":
                state = "up"
                count += 1

        # Tampilkan Mode, Jumlah Repetisi (Count), dan State posisi
        cv2.putText(
            img, f"Mode: {MODE.upper()}  Count: {count}", (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2
        )
        cv2.putText(
            img, f"State: {state}", (20, 100),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2
        )

        cv2.imshow(window_name, img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
        if key == ord('m'):
            MODE = "pushup" if MODE == "squat" else "squat"
finally:
    cap.release()
    cv2.destroyAllWindows()


#Penugasan 01 -

#Penugasan 02 -

#Penugasan 03 -

#Penugasan 04 -